In [1]:
import pandas as pd

# Caricamento: separatore ; (formato europeo) e quoting per gestire i campi con newline interni
df = pd.read_csv("aviation_raw.csv", sep=";", quotechar='"', engine="python")

print("Dimensioni:", df.shape)
print("Memoria:", round(df.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
df.head(3)

Dimensioni: (7462, 48)
Memoria: 169.1 MB


,NtsbNo,EventType,Mkey,EventDate,City,State,Country,ReportNo,N#,SerialNumber,...,WeatherCondition,Operator,BroadPhaseofFlight,ReportStatus,RepGenFlag,MostRecentReportType,DocketUrl,ReportUrl,rep_text,rep_num_jsg
0,ERA24LA084,ACC,193603,2023-12-31T15:13:00Z,Midland,Virginia,United States,NaN,N37GA,004CE,...,VMC,NaN,Landing,Completed,NaN,Final,https://data.ntsb.gov/Docket?ProjectID=193603,https://data.ntsb.gov/carol-repgen/api/Aviatio...,Page 1 of 5\nAviation Investigation Final Repo...,ERA24LA084
1,ERA24LA079,ACC,193587,2023-12-30T15:04:00Z,Daytona Beach,Florida,United States,NaN,"N828AK, FA3XNWMRAN","1689, 163DF81001N020",...,VMC,Tunica Helicopters LLC,Approach,Completed,NaN,Final,NaN,https://data.ntsb.gov/carol-repgen/api/Aviatio...,Page 1 of 7\nAviation Investigation Final Repo...,ERA24LA079
2,CEN24LA081,ACC,193605,2023-12-29T15:27:00Z,Beaumont,Texas,United States,NaN,N71MS,7452C,...,VMC,TAF AERIAL SERVICES LLC,Landing,Completed,NaN,Final,https://data.ntsb.gov/Docket?ProjectID=193605,https://data.ntsb.gov/carol-repgen/api/Aviatio...,Page 1 of 5\nAviation Investigation Final Repo...,CEN24LA081


In [2]:
# Riepilogo: tipo dato, valori non-null, percentuale di NaN per colonna
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null_%": (df.isnull().sum() / len(df) * 100).round(1)
})
summary

,dtype,non_null,null_%
NtsbNo,str,7462,0.0
EventType,str,7462,0.0
Mkey,int64,7462,0.0
EventDate,str,7462,0.0
City,str,7462,0.0
State,str,7438,0.3
Country,str,7462,0.0
ReportNo,str,19,99.7
N#,str,7462,0.0
SerialNumber,str,7459,0.0


In [3]:
# Colonne da droppare per 3 motivi diversi:
columns_to_drop = [
    # 1) Morte (100% NaN): non aggiungono nulla
    "EventID", "RepGenFlag",
    
    # 2) Quasi morte (>99% NaN): irrilevanti statisticamente
    "ReportNo",
    
    # 3) Report PDF e metadati pesanti: irrilevanti per dashboard, mangiano memoria
    "rep_text", "rep_num_jsg", "ProbableCause", "Findings",
    "DocketUrl", "ReportUrl",
    
    # 4) Identificativi tecnici interni NTSB: non servono per analisi
    "Mkey", "SerialNumber", "N#",
    
    # 5) Date/flag di reporting: irrilevanti per il safety storytelling
    "OriginalPublishedDate", "DocketOriginalPublishedDate",
    "ReportStatus", "MostRecentReportType", "ReportType",
    "HasSafetyRec", "Mode",
    
    # 6) Aeroporto: 30% NaN e in dashboard useremo Lat/Long invece
    "AirportID", "AirportName"
]

df = df.drop(columns=columns_to_drop)

print("Nuove dimensioni:", df.shape)
print("Memoria:", round(df.memory_usage(deep=True).sum() / 1024**2, 2), "MB")
print("\nColonne rimaste:")
for c in df.columns:
    print(f"  - {c}")

Nuove dimensioni: (7462, 27)
Memoria: 8.02 MB

Colonne rimaste:
  - NtsbNo
  - EventType
  - EventDate
  - City
  - State
  - Country
  - HighestInjuryLevel
  - FatalInjuryCount
  - SeriousInjuryCount
  - MinorInjuryCount
  - OnboardInjuryCount
  - OnGroundInjuryCount
  - Latitude
  - Longitude
  - Make
  - Model
  - AirCraftCategory
  - AmateurBuilt
  - NumberOfEngines
  - EngineType
  - Scheduled
  - PurposeOfFlight
  - FAR
  - AirCraftDamage
  - WeatherCondition
  - Operator
  - BroadPhaseofFlight


In [4]:
# Conversione a datetime — ISO 8601 con suffisso Z (UTC)
df["EventDate"] = pd.to_datetime(df["EventDate"], format="ISO8601", utc=True)

# Estrazione componenti temporali per analisi temporale in dashboard
df["Year"] = df["EventDate"].dt.year
df["Month"] = df["EventDate"].dt.month
df["Quarter"] = df["EventDate"].dt.quarter
df["YearMonth"] = df["EventDate"].dt.to_period("M").astype(str)  # es: "2023-12"

# Verifica
print("Range temporale:", df["EventDate"].min(), "→", df["EventDate"].max())
print("\nIncidenti per anno:")
print(df["Year"].value_counts().sort_index())

Range temporale: 2016-01-01 15:30:00+00:00 → 2023-12-31 15:13:00+00:00

Incidenti per anno:
Year
2016    1343
2017    1330
2018    1351
2019    1308
2022    1256
2023     874
Name: count, dtype: int64


C:\Users\FedeO\AppData\Local\Temp\ipykernel_24520\866395500.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["YearMonth"] = df["EventDate"].dt.to_period("M").astype(str)  # es: "2023-12"


In [5]:
# Sopprime il warning su timezone (corretto perché un Period mensile non ha senso con TZ)
df["YearMonth"] = df["EventDate"].dt.tz_localize(None).dt.to_period("M").astype(str)

# Stagionalità: incidenti per mese (su tutti gli anni aggregati)
print("Incidenti per mese (aggregato 2016-2023):")
print(df["Month"].value_counts().sort_index())

Incidenti per mese (aggregato 2016-2023):
Month
1     395
2     414
3     529
4     554
5     750
6     891
7     966
8     818
9     705
10    603
11    436
12    401
Name: count, dtype: int64


In [6]:
# Vediamo i valori unici e le frequenze
print("Valori unici di HighestInjuryLevel:")
print(df["HighestInjuryLevel"].value_counts(dropna=False))

print("\nIncrocio: HighestInjuryLevel NaN vs FatalInjuryCount")
nan_mask = df["HighestInjuryLevel"].isna()
print(df.loc[nan_mask, ["FatalInjuryCount", "SeriousInjuryCount", "MinorInjuryCount"]].describe())

Valori unici di HighestInjuryLevel:
HighestInjuryLevel
NaN        4259
Minor      1176
Fatal      1123
Serious     904
Name: count, dtype: int64

Incrocio: HighestInjuryLevel NaN vs FatalInjuryCount
       FatalInjuryCount  SeriousInjuryCount  MinorInjuryCount
count            4246.0              4246.0       4246.000000
mean                0.0                 0.0          0.000707
std                 0.0                 0.0          0.034313
min                 0.0                 0.0          0.000000
25%                 0.0                 0.0          0.000000
50%                 0.0                 0.0          0.000000
75%                 0.0                 0.0          0.000000
max                 0.0                 0.0          2.000000


In [7]:
# I NaN sono incidenti senza feriti → imputiamo con "None"
df["HighestInjuryLevel"] = df["HighestInjuryLevel"].fillna("None")

# Lo trasformiamo in categoria ordinale (None < Minor < Serious < Fatal)
# Importante per i grafici: Power BI rispetta l'ordine se è categoriale
injury_order = ["None", "Minor", "Serious", "Fatal"]
df["HighestInjuryLevel"] = pd.Categorical(
    df["HighestInjuryLevel"], 
    categories=injury_order, 
    ordered=True
)

# Verifica
print(df["HighestInjuryLevel"].value_counts().sort_index())
print("\nTipo:", df["HighestInjuryLevel"].dtype)

HighestInjuryLevel
None       4259
Minor      1176
Serious     904
Fatal      1123
Name: count, dtype: int64

Tipo: category


In [8]:
# Total injuries: somma di tutti i tipi di feriti (onboard + on ground)
injury_cols = ["FatalInjuryCount", "SeriousInjuryCount", "MinorInjuryCount", "OnGroundInjuryCount"]

# Riempiamo i NaN con 0 prima di sommare (NaN + 0 = NaN, vorremmo 0)
df[injury_cols] = df[injury_cols].fillna(0)
df["TotalInjuries"] = df[injury_cols].sum(axis=1)

# Verifica
print("Distribuzione TotalInjuries:")
print(df["TotalInjuries"].describe())
print(f"\nIncidenti con almeno 1 ferito: {(df['TotalInjuries'] > 0).sum()}")
print(f"Incidente più grave: {df['TotalInjuries'].max():.0f} feriti totali")

Distribuzione TotalInjuries:
count    7462.000000
mean        0.759582
std         2.000693
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max       134.000000
Name: TotalInjuries, dtype: float64

Incidenti con almeno 1 ferito: 3200
Incidente più grave: 134 feriti totali


In [9]:
worst = df.loc[df["TotalInjuries"].idxmax()]
print(f"Data: {worst['EventDate']}")
print(f"Luogo: {worst['City']}, {worst['State']}")
print(f"Aeromobile: {worst['Make']} {worst['Model']}")
print(f"Operatore: {worst['Operator']}")
print(f"Fatali: {worst['FatalInjuryCount']:.0f}, Gravi: {worst['SeriousInjuryCount']:.0f}, Lievi: {worst['MinorInjuryCount']:.0f}, A terra: {worst['OnGroundInjuryCount']:.0f}")

Data: 2018-04-17 11:03:00+00:00
Luogo: Philadelphia, Pennsylvania
Aeromobile: BOEING 737 7H4
Operatore: SOUTHWEST AIRLINES CO
Fatali: 1, Gravi: 8, Lievi: 125, A terra: 0


In [11]:
print("Scheduled — valori unici:")
print(df["Scheduled"].value_counts(dropna=False))

print("\nPurposeOfFlight — valori unici:")
print(df["PurposeOfFlight"].value_counts(dropna=False))

Scheduled — valori unici:
Scheduled
NaN           6688
NSCH           528
SCHD           219
SCHD, SCHD      19
NSCH, NSCH       5
NSCH, SCHD       2
SCHD, NSCH       1
Name: count, dtype: int64

PurposeOfFlight — valori unici:
PurposeOfFlight
PERS          4646
INST          1090
NaN            411
AAPL           328
BUS            205
POSI           121
AOBV           108
OWRK           107
FLTS            99
FERY            41
SKYD            34
PERS, PERS      33
EXLD            27
ASHO            22
PUBF            21
BANT            19
UNK             18
PUBL            18
EXEC            14
PUBS            13
FIRF            12
INST, INST      12
INST, PERS      11
GLDT            11
PUBU            10
PERS, INST       6
ASHO, ASHO       6
AAPL, AAPL       4
BUS, BUS         4
ADRP             2
OWRK, OWRK       1
POSI, BUS        1
INST, PUBF       1
INST, EXEC       1
AOBV, AOBV       1
BUS, INST        1
POSI, PERS       1
POSI, POSI       1
PERS, BUS        1
Name: count, dt

In [12]:
def classify_segment(row):
    """
    Classifica ogni incidente in 5 macro-segmenti operativi.
    Logica: prima controlla Scheduled (rivela voli di linea), 
    poi usa PurposeOfFlight per dettagliare il resto.
    """
    scheduled = str(row["Scheduled"]).split(",")[0].strip() if pd.notna(row["Scheduled"]) else ""
    purpose = str(row["PurposeOfFlight"]).split(",")[0].strip() if pd.notna(row["PurposeOfFlight"]) else ""
    
    # Voli di linea schedulati: massima visibilità mediatica e regolatoria
    if scheduled == "SCHD":
        return "Commercial Scheduled"
    
    # Voli non schedulati ma con codice commerciale → charter, taxi aereo
    if scheduled == "NSCH":
        return "Commercial Non-Scheduled"
    
    # Da qui in poi non c'è "Scheduled" definito → general aviation
    if purpose == "PERS":
        return "Personal"
    if purpose == "INST":
        return "Instructional"
    if purpose == "BUS":
        return "Business"
    if purpose in ["AAPL", "AOBV", "OWRK", "FLTS", "FERY", "SKYD", "POSI"]:
        return "Aerial Work"
    
    return "Other / Unknown"

df["OperationSegment"] = df.apply(classify_segment, axis=1)

# Verifica distribuzione
print("Distribuzione per OperationSegment:")
print(df["OperationSegment"].value_counts())
print(f"\nTotale classificato: {(df['OperationSegment'] != 'Other / Unknown').sum()} / {len(df)}")

Distribuzione per OperationSegment:
OperationSegment
Personal                    4495
Instructional               1072
Aerial Work                  779
Commercial Non-Scheduled     535
Commercial Scheduled         239
Other / Unknown              175
Business                     167
Name: count, dtype: int64

Totale classificato: 7287 / 7462


In [13]:
# Normalizza Make: tutto maiuscolo, rimuove suffissi aziendali frequenti
df["Make"] = (df["Make"]
    .str.upper()
    .str.replace(r"\s+(INC|LLC|CORP|CO|COMPANY|AIRCRAFT|AVIATION|LTD)\.?$", "", regex=True)
    .str.strip()
)

# Filtra agli USA per analisi geo coerente
print(f"Prima del filtro USA: {len(df)} righe")
df = df[df["Country"] == "United States"].reset_index(drop=True)
print(f"Dopo il filtro USA: {len(df)} righe")

# Top 10 Make dopo la pulizia
print("\nTop 10 costruttori per numero di incidenti:")
print(df["Make"].value_counts().head(10))

Prima del filtro USA: 7462 righe
Dopo il filtro USA: 7438 righe

Top 10 costruttori per numero di incidenti:
Make
CESSNA                 1848
PIPER                  1116
BEECH                   402
BELL                    165
ROBINSON HELICOPTER     150
AIR TRACTOR             122
BOEING                  118
MOONEY                   91
CIRRUS DESIGN            79
ROBINSON                 77
Name: count, dtype: int64


In [14]:
# Robinson Helicopter è registrato in due varianti — accorpiamo
df["Make"] = df["Make"].replace({"ROBINSON HELICOPTER": "ROBINSON"})

# Verifica
print(df["Make"].value_counts().head(10))

Make
CESSNA           1848
PIPER            1116
BEECH             402
ROBINSON          227
BELL              165
AIR TRACTOR       122
BOEING            118
MOONEY             91
CIRRUS DESIGN      79
VANS               69
Name: count, dtype: int64


In [15]:
# Le colonne raw che abbiamo già processato in feature derivate non servono più
columns_to_drop_final = [
    "Country",          # Tutti USA dopo il filtro, colonna costante = inutile
    "Scheduled",        # Già inglobata in OperationSegment
    "PurposeOfFlight",  # Idem
    "EventType",        # Sempre "ACC" (Accident) o "INC" (Incident), poco utile per dashboard
    "FAR",              # Codice regolatorio FAA, troppo tecnico
    "AmateurBuilt",     # Quasi sempre "false", poco informativo
    "NumberOfEngines",  # Stringa con virgole multi-aereo, troppo sporca
]

df = df.drop(columns=columns_to_drop_final)

# Riepilogo finale
print(f"Dimensioni finali: {df.shape}")
print(f"Memoria: {round(df.memory_usage(deep=True).sum() / 1024**2, 2)} MB")
print(f"\nColonne finali ({len(df.columns)}):")
for c in df.columns:
    print(f"  - {c}  ({df[c].dtype})")

Dimensioni finali: (7438, 26)
Memoria: 5.71 MB

Colonne finali (26):
  - NtsbNo  (str)
  - EventDate  (datetime64[us, UTC])
  - City  (str)
  - State  (str)
  - HighestInjuryLevel  (category)
  - FatalInjuryCount  (float64)
  - SeriousInjuryCount  (float64)
  - MinorInjuryCount  (float64)
  - OnboardInjuryCount  (float64)
  - OnGroundInjuryCount  (float64)
  - Latitude  (float64)
  - Longitude  (float64)
  - Make  (str)
  - Model  (str)
  - AirCraftCategory  (str)
  - EngineType  (str)
  - AirCraftDamage  (str)
  - WeatherCondition  (str)
  - Operator  (str)
  - BroadPhaseofFlight  (str)
  - Year  (int32)
  - Month  (int32)
  - Quarter  (int32)
  - YearMonth  (str)
  - TotalInjuries  (float64)
  - OperationSegment  (str)


In [16]:
df.to_csv(
    "aviation_clean.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"  # con BOM, così Power BI riconosce caratteri accentati
)

print("File salvato: aviation_clean.csv")
print(f"Righe esportate: {len(df)}")
print(f"Colonne esportate: {len(df.columns)}")

File salvato: aviation_clean.csv
Righe esportate: 7438
Colonne esportate: 26
